# 第三模块：预测标签与模型数据预处理

顺序：`data_loader.ipynb` → `feature.ipynb` → **本 notebook** → 后续因子评价 / 模型训练。
输入为 `data/processed/alpha158` 的成功构建结果；输出为 `data/processed/model_dataset`。
本模块不重新计算 Alpha158，不执行其他 notebook，也不修改原始数据和上游产物。

本模块完成：选择预测目标、验证标签日期、时间切分与跨界清除、训练期拟合预处理、
分批导出、质量报告及模型数据加载。输入始终保持标准 158 列及其顺序。
上游没有成功输出时，正式处理跳过，独立自检仍可运行。

`feature.ipynb` 已在**完整交易日历**上计算 1 / 5 / 10 日收益。
本模块复用并验证这些原始标签，不能对预热、停牌过滤后的因子行直接 `shift` 生成收益。
默认 `LABEL0 = close[t+2] / close[t+1] - 1`，t 日收盘后产生特征。
这是收盘到收盘研究标签，交易模块仍需保持相同成交口径。

## 1. 配置：数据源、目标、切分和预处理

必需依赖：NumPy、pandas；可选 pyarrow（Parquet），无 pyarrow 时使用 gzip CSV。
`SOURCE_RUN=None` 时，启动时读取一次 `alpha158/latest_run.json`，并固定该上游版本。
也可指定一个历史 `run_...` 目录，以复现实验。

`TARGET_MODE='return'` 使用原始收益做回归；`'direction'` 以收益是否超过阈值做二分类。
二分类缺标签保留 NaN，不把未知未来收益误记为 0。`TARGET_HORIZON=1/5/10` 分别选择
`LABEL0/LABEL5/LABEL10`。使用上游未生成的周期或不同进入延迟时，应先调整并重跑上游。

固定时间切分供首轮模型对照；本模块不随机划分、不进行模型调参或因子筛选。

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import importlib.util
import json
import shutil
import uuid
import warnings
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 12)
PROJECT_DIR = next((p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / 'feature.ipynb').exists() and (p / 'data').is_dir()),
                   Path(r'E:\document-prefer\python for finanace\multi-model-bactest'))
FEATURE_ROOT = PROJECT_DIR / 'data' / 'processed' / 'alpha158'
RAW_DIR = PROJECT_DIR / 'data' / 'raw'
OUTPUT_ROOT = PROJECT_DIR / 'data' / 'processed' / 'model_dataset'
SOURCE_RUN = None
TARGET_HORIZON = 1
ENTRY_LAG = 1
TARGET_MODE = 'return'          # 'return' 或 'direction'
DIRECTION_THRESHOLD = 0.0
SPLITS = {
    'train': ('2022-01-01', '2023-12-31'),
    'valid': ('2024-01-01', '2024-12-31'),
    'test': ('2025-01-01', '2025-12-31'),
}
INSTRUMENTS = None              # None 为上游股票全集；或 ['000001.SZ', '600000.SH']
MAX_ROW_MISSING = 0.50         # 原始 158 列缺失超过此比例，保留记录但不进入模型
CLIP_QUANTILES = (0.01, 0.99)  # 仅 linear 版本截尾；None 关闭截尾
FIT_MAX_ROWS = 100_000         # 训练样本的确定性抽样上限；None 使用全部训练行
OUTPUT_FORMAT = 'parquet' if importlib.util.find_spec('pyarrow') else 'csv.gz'
RUN_PROCESS = True
SCHEMA_VERSION = 1
print(f'项目：{PROJECT_DIR}\n目标：{TARGET_MODE}，持有 {TARGET_HORIZON} 日，进入延迟 {ENTRY_LAG} 日')

## 2. 固定输入版本并检查契约

必须有 `metadata.json`、`feature_names.json` 和完整分区。
标签周期、进入延迟、158 列名称及分区数量都检查一致性。
上游原有 `split` / `*_usable` 只作历史记录，本模块按照当前配置重新计算。
本次处理的上游路径、元数据哈希、分区大小和修改时间会写入输出，以追踪来源。

In [ ]:
KBAR = ['KMID', 'KLEN', 'KMID2', 'KUP', 'KUP2', 'KLOW', 'KLOW2', 'KSFT', 'KSFT2']
ROLLING = ['ROC', 'MA', 'STD', 'BETA', 'RSQR', 'RESI', 'MAX', 'MIN', 'QTLU', 'QTLD',
           'RANK', 'RSV', 'IMAX', 'IMIN', 'IMXD', 'CORR', 'CORD', 'CNTP', 'CNTN',
           'CNTD', 'SUMP', 'SUMN', 'SUMD', 'VMA', 'VSTD', 'WVMA', 'VSUMP', 'VSUMN', 'VSUMD']
EXPECTED_FEATURES = KBAR + ['OPEN0', 'HIGH0', 'LOW0', 'VWAP0'] + [
    f'{name}{window}' for name in ROLLING for window in (5, 10, 20, 30, 60)]

def label_name(horizon):
    return 'LABEL0' if horizon == 1 else f'LABEL{horizon}'

def validate_settings(config):
    splits = config['splits']
    if list(splits) != ['train', 'valid', 'test']:
        raise ValueError('splits 需按 train、valid、test 顺序给出')
    previous_end = None
    for name, (start, end) in splits.items():
        start, end = pd.Timestamp(start), pd.Timestamp(end)
        if start > end or (previous_end is not None and start <= previous_end):
            raise ValueError('时间区间不可重叠，且必须按时间递增')
        previous_end = end
    if config['target_mode'] not in ('return', 'direction'):
        raise ValueError('target_mode 必须为 return 或 direction')
    for name in ('target_horizon', 'entry_lag'):
        value = config[name]
        if isinstance(value, bool) or not isinstance(value, int) or value < 1:
            raise ValueError(f'{name} 必须为正整数')
    if not np.isfinite(config['direction_threshold']):
        raise ValueError('涨跌分类阈值必须为有限数值')
    if not 0 <= config['max_row_missing'] <= 1:
        raise ValueError('max_row_missing 必须位于 [0,1]')
    quantiles = config['clip_quantiles']
    if quantiles is not None and not 0 <= quantiles[0] < quantiles[1] <= 1:
        raise ValueError('截尾分位数配置无效')
    limit = config['fit_max_rows']
    if limit is not None and (isinstance(limit, bool) or not isinstance(limit, int) or limit < 2):
        raise ValueError('fit_max_rows 需为至少 2 的整数或 None')
    if config['output_format'] not in ('parquet', 'csv.gz'):
        raise ValueError('不支持的输出格式')

def make_config():
    config = {
        'target_horizon': TARGET_HORIZON, 'entry_lag': ENTRY_LAG,
        'target_mode': TARGET_MODE, 'direction_threshold': DIRECTION_THRESHOLD,
        'splits': {key: list(value) for key, value in SPLITS.items()},
        'instruments': INSTRUMENTS, 'max_row_missing': MAX_ROW_MISSING,
        'clip_quantiles': list(CLIP_QUANTILES) if CLIP_QUANTILES is not None else None,
        'fit_max_rows': FIT_MAX_ROWS, 'output_format': OUTPUT_FORMAT,
    }
    validate_settings(config)
    return config

def resolve_source(root, run=None):
    root = Path(root)
    if run is not None:
        path = Path(run)
        return path if path.is_absolute() else root / path
    pointer = root / 'latest_run.json'
    if not pointer.exists():
        raise FileNotFoundError('没有 Alpha158 成功输出；先运行 feature.ipynb 第 7 节')
    name = json.loads(pointer.read_text(encoding='utf-8'))['run_dir']
    path = (root / name).resolve()
    if path.parent != root.resolve():
        raise ValueError('latest_run.json 必须指向 alpha158 目录内的一个运行目录')
    return path

def inspect_source(path, config):
    path = Path(path).resolve()
    metadata_bytes = (path / 'metadata.json').read_bytes()
    metadata = json.loads(metadata_bytes)
    names = json.loads((path / 'feature_names.json').read_text(encoding='utf-8'))
    if names != EXPECTED_FEATURES or metadata.get('features') != 158:
        raise ValueError('上游不是当前标准 Alpha158 的 158 列及顺序')
    if config['target_horizon'] not in metadata['label_horizons']:
        raise ValueError('所选持有期不在上游 label_horizons；请先重建 feature 数据')
    if config['entry_lag'] != metadata['label_entry_lag']:
        raise ValueError('标签进入延迟与上游不同；请先调整并重建 feature 数据')
    source_format = metadata['format']
    if source_format not in ('parquet', 'csv.gz'):
        raise ValueError('不支持的上游格式')
    pattern = 'part-*.parquet' if source_format == 'parquet' else 'part-*.csv.gz'
    paths = sorted(path.glob(pattern))
    if not paths or len(paths) != metadata['partitions']:
        raise ValueError('上游分区缺失或分区数量不符')
    manifest = [{'file': p.name, 'bytes': p.stat().st_size, 'mtime_ns': p.stat().st_mtime_ns}
                for p in paths]
    if any(item['bytes'] == 0 for item in manifest):
        raise ValueError('上游存在空分区')
    return {'path': path, 'metadata': metadata, 'features': names, 'paths': paths,
            'metadata_sha256': hashlib.sha256(metadata_bytes).hexdigest(), 'manifest': manifest}

config = make_config()
source = None
try:
    source = inspect_source(resolve_source(FEATURE_ROOT, SOURCE_RUN), config)
    display(pd.Series({key: source['metadata'][key] for key in ('rows', 'instruments', 'partitions', 'format')}))
    print(f'已固定上游：{source["path"]}')
except FileNotFoundError as exc:
    print(f'上游尚未就绪：{exc}。可以继续运行自检，正式处理将跳过。')

## 3. 标签审计与时间切分

从上游输入范围内的交易日历验证 `label_entry_date` 和所选 `LABEL*_end_date`。
不按自然日推算，不把复牌日当下一交易日。
对每行重新记录：原始收益 `y_raw`、模型目标 `target`、缺失率、所属分段及可用标记。

有监督样本需同时满足：特征缺失比例不过限、标签存在、标签结束日期不晚于所属分段末日。
不满足条件的记录仍保存，方便审计；最新日期缺标签时仍可用于预测。
`outside` 行不用于拟合或正式模型数据输出。切分间可留时间间隔。

In [ ]:
def load_calendar(raw_dir, start, end):
    paths = sorted((Path(raw_dir) / 'basic').glob('trade_dates_*.csv'))
    if not paths:
        raise FileNotFoundError('缺少 data/raw/basic/trade_dates_*.csv')
    values = []
    for path in paths:
        values.extend(pd.read_csv(path, dtype=str, usecols=['trade_date']).trade_date.tolist())
    calendar = pd.DatetimeIndex(pd.to_datetime(values, format='%Y%m%d', errors='raise')).unique().sort_values()
    calendar = calendar[(calendar >= pd.Timestamp(start)) & (calendar <= pd.Timestamp(end))]
    if calendar.empty:
        raise ValueError('上游日期范围内没有交易日历')
    return calendar

def read_feature_partition(path, source):
    name = label_name(source['selected_horizon'])
    required = source['features'] + [name, 'label_entry_date', f'{name}_end_date']
    if source['metadata']['format'] == 'parquet':
        frame = pd.read_parquet(path, columns=required)
    else:
        frame = pd.read_csv(path, usecols=['datetime', 'instrument'] + required,
                            dtype={'instrument': str})
        frame['datetime'] = pd.to_datetime(frame['datetime'], errors='raise')
        frame = frame.set_index(['datetime', 'instrument'])
    if frame.index.names != ['datetime', 'instrument']:
        raise ValueError(f'索引契约不符：{path}')
    dates = pd.to_datetime(frame.index.get_level_values('datetime'), errors='raise')
    instruments = frame.index.get_level_values('instrument')
    if dates.isna().any() or pd.isna(instruments).any() or (instruments.astype(str) == '').any():
        raise ValueError('日期或股票代码为空')
    frame.index = pd.MultiIndex.from_arrays([dates, instruments.astype(str)], names=['datetime', 'instrument'])
    if frame.index.has_duplicates:
        raise ValueError(f'分区内日期股票主键重复：{path}')
    for column in source['features'] + [name]:
        frame[column] = pd.to_numeric(frame[column], errors='raise')
    for column in ('label_entry_date', f'{name}_end_date'):
        frame[column] = pd.to_datetime(frame[column], errors='raise')
    if np.isinf(frame[name].to_numpy(dtype=float)).any():
        raise ValueError('收益标签出现无穷值')
    return frame.sort_index()

def target_and_flags(frame, names, calendar, config):
    name = label_name(config['target_horizon'])
    dates = pd.DatetimeIndex(frame.index.get_level_values('datetime'))
    if not dates.isin(calendar).all():
        raise ValueError('因子日期不在上游交易日历中')
    date_series = pd.Series(calendar, index=calendar)
    for field, lag in [('label_entry_date', config['entry_lag']),
                       (f'{name}_end_date', config['entry_lag'] + config['target_horizon'])]:
        expected = pd.Series(date_series.shift(-lag).reindex(dates).to_numpy(), index=frame.index)
        actual = pd.to_datetime(frame[field])
        equal = actual.eq(expected) | (actual.isna() & expected.isna())
        if not equal.all():
            raise ValueError(f'{field} 与交易日历/标签配置不符')
    X = frame[names].replace([np.inf, -np.inf], np.nan).astype('float32')
    info = pd.DataFrame(index=frame.index)
    info['y_raw'] = frame[name].astype(float)
    info['label_entry_date'] = frame.label_entry_date
    info['label_end_date'] = frame[f'{name}_end_date']
    invalid_label = info.y_raw.notna() & (info.label_entry_date.isna() | info.label_end_date.isna())
    if invalid_label.any():
        raise ValueError('存在有收益却缺失进入/结束日期的标签')
    if config['target_mode'] == 'return':
        info['target'] = info.y_raw
    else:
        info['target'] = info.y_raw.gt(config['direction_threshold']).astype(float).where(info.y_raw.notna())
    info['missing_ratio'] = X.isna().mean(axis=1)
    info['feature_eligible'] = info.missing_ratio.le(config['max_row_missing'])
    info['split'] = 'outside'
    boundary = pd.Series(pd.NaT, index=info.index, dtype='datetime64[ns]')
    for split, (start, end) in config['splits'].items():
        mask = (dates >= pd.Timestamp(start)) & (dates <= pd.Timestamp(end))
        info.loc[mask, 'split'] = split
        boundary.loc[mask] = pd.Timestamp(end)
    info['label_eligible'] = info.target.notna() & info.label_end_date.le(boundary)
    info['supervised_eligible'] = info.feature_eligible & info.label_eligible
    # 被清除的原始标签可审计，但不会作为监督目标暴露。
    info['target'] = info.target.where(info.supervised_eligible)
    info['label_status'] = np.select(
        [info['split'].eq('outside'), info.y_raw.isna(), info.label_end_date.gt(boundary)],
        ['outside', 'missing_label', 'cross_boundary'], default='ok')
    return X, info

def iter_source(source, calendar, config):
    source = dict(source, selected_horizon=config['target_horizon'])
    # 上游按股票分批：同一股票不应出现在两个分区，避免跨分区主键重复。
    seen_instruments = set()
    for path in source['paths']:
        frame = read_feature_partition(path, source)
        present = set(frame.index.get_level_values('instrument'))
        overlap = present & seen_instruments
        if overlap:
            raise ValueError(f'上游跨分区股票重复：{sorted(overlap)[:3]}')
        seen_instruments.update(present)
        if config['instruments'] is not None:
            frame = frame.loc[frame.index.get_level_values('instrument').isin(config['instruments'])]
        X, info = target_and_flags(frame, source['features'], calendar, config)
        yield path, X, info

## 4. 仅在训练期拟合预处理

为控制全市场内存，默认从**训练期且标签可用、特征缺失不过限**的样本中，
按 `(datetime, instrument)` 的稳定哈希选最多 100,000 行拟合统计量。
因此中位数和分位数是训练抽样估计；设置 `FIT_MAX_ROWS=None` 可使用全部训练样本，内存需求会增加。
验证、测试和预测数据均只应用已保存的参数，不重新拟合。修改验证或测试值不会改变拟合参数。

- `tree`：训练中位数填补，不缩放、不截尾，适用于树模型。
- `linear`：训练分位数截尾 → 训练中位数填补 → 训练均值/标准差缩放。

全空训练列使用 0 填补，常数列缩放分母设为 1，仍保留 158 列。
原始收益标签不做标准化，便于不同模型的预测评价。

In [ ]:
def collect_fit_sample(source, calendar, config):
    sample = pd.DataFrame()
    train_rows = 0
    audit = []
    total_source_rows = 0
    for path, X, info in iter_source(source, calendar, config):
        total_source_rows += len(info)
        mask = info['split'].eq('train') & info.supervised_eligible
        selected = X.loc[mask].copy()
        train_rows += len(selected)
        sample = sample_training_rows(sample, selected, config['fit_max_rows'])
        counts = info.groupby('split', sort=False).agg(
            rows=('split', 'size'), feature_eligible=('feature_eligible', 'sum'),
            label_eligible=('label_eligible', 'sum'), supervised_eligible=('supervised_eligible', 'sum'))
        counts['missing_labels'] = info.y_raw.isna().groupby(info['split']).sum()
        counts['purged_labels'] = info.label_status.eq('cross_boundary').groupby(info['split']).sum()
        audit.append(counts)
    # instruments=None 才能和上游全量行数对比。
    if config['instruments'] is None and total_source_rows != source['metadata']['rows']:
        raise ValueError('读取行数与上游 metadata.rows 不符')
    if sample.empty:
        raise ValueError('没有训练期可用标签样本，无法拟合预处理')
    summary = pd.concat(audit).groupby(level=0).sum().reindex(['train', 'valid', 'test', 'outside'], fill_value=0)
    if any(summary.loc[split, 'supervised_eligible'] == 0 for split in ('train', 'valid', 'test')):
        display(summary)
        raise ValueError('train/valid/test 均需有可用监督样本，请检查上游覆盖、日期和标签周期')
    sample = sample.drop(columns='_sample_hash').sort_index()
    return sample, train_rows, summary

def finite_list(series):
    return [float(value) if pd.notna(value) and np.isfinite(value) else None for value in series]

def fit_preprocessors(sample, config):
    if sample.empty or list(sample) != EXPECTED_FEATURES:
        raise ValueError('拟合样本为空或特征顺序错误')
    sample = sample.replace([np.inf, -np.inf], np.nan).astype(float)
    median = sample.median().fillna(0.)
    quantiles = config['clip_quantiles']
    low = sample.quantile(quantiles[0]) if quantiles is not None else pd.Series(np.nan, index=sample.columns)
    high = sample.quantile(quantiles[1]) if quantiles is not None else pd.Series(np.nan, index=sample.columns)
    clipped = sample.clip(lower=low.fillna(-np.inf), upper=high.fillna(np.inf), axis=1)
    linear_median = clipped.median().fillna(0.)
    filled = clipped.fillna(linear_median)
    mean, scale = filled.mean(), filled.std(ddof=0)
    constant = scale.le(1e-12) | scale.isna()
    scale = scale.mask(constant, 1.)
    return {
        'schema_version': SCHEMA_VERSION, 'features': list(sample), 'fit_rows': len(sample),
        'tree_median': finite_list(median), 'linear_low': finite_list(low),
        'linear_high': finite_list(high), 'linear_median': finite_list(linear_median),
        'linear_mean': finite_list(mean), 'linear_scale': finite_list(scale),
        'all_missing_features': sample.columns[sample.isna().all()].tolist(),
        'constant_linear_features': sample.columns[constant].tolist(),
        'fit_index_sha256': hashlib.sha256(pd.util.hash_pandas_object(sample.index, index=True).to_numpy().tobytes()).hexdigest(),
    }

def transform_features(X, state, view='tree'):
    if list(X) != state['features']:
        raise ValueError('特征名称或列顺序与拟合状态不一致')
    X = X.replace([np.inf, -np.inf], np.nan).astype(float)
    def parameter(key):
        return pd.Series(state[key], index=state['features'], dtype=float)
    if view == 'tree':
        result = X.fillna(parameter('tree_median'))
    elif view == 'linear':
        result = X.clip(lower=parameter('linear_low').fillna(-np.inf),
                        upper=parameter('linear_high').fillna(np.inf), axis=1)
        result = (result.fillna(parameter('linear_median')) - parameter('linear_mean')) / parameter('linear_scale')
    else:
        raise ValueError('view 必须为 tree 或 linear')
    result = result.astype('float32')
    if not np.isfinite(result.to_numpy()).all():
        raise ValueError('预处理后仍存在非有限值，请检查数值范围和训练统计量')
    return result

def sample_training_rows(sample, selected, limit):
    if selected.empty:
        return sample
    selected = selected.copy()
    selected['_sample_hash'] = pd.util.hash_pandas_object(selected.index, index=True).to_numpy()
    if limit is not None:
        selected = selected.nsmallest(limit, '_sample_hash', keep='first')
    sample = pd.concat([sample, selected])
    if limit is not None and len(sample) > limit:
        sample = sample.nsmallest(limit, '_sample_hash', keep='first')
    return sample

## 5. 分批输出和质量报告

每个运行保存 `config.json`、`source.json`、`preprocessor.json`、`metadata.json`、
`feature_names.json`、`split_summary.csv`、`training_feature_report.csv`，
并按 `tree/linear → train/valid/test → part-*` 保存数据。
特征报告来自拟合训练样本；分段统计来自全部所选输入行。

同一上游快照在两次读取之间发生变化会报错。仅成功构建后更新 `latest_run.json`，
失败会清理本次临时目录，已有运行不覆盖。分类阈值、截尾和抽样参数均保存在配置中。

In [ ]:
def assert_source_unchanged(source):
    for item in source['manifest']:
        path = source['path'] / item['file']
        stat = path.stat()
        if stat.st_size != item['bytes'] or stat.st_mtime_ns != item['mtime_ns']:
            raise RuntimeError(f'上游分区在处理中发生变化：{path}')
    digest = hashlib.sha256((source['path'] / 'metadata.json').read_bytes()).hexdigest()
    if digest != source['metadata_sha256']:
        raise RuntimeError('上游元数据在处理中发生变化')

def write_json(path, data):
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')

def write_table(frame, path, output_format):
    if output_format == 'parquet':
        frame.to_parquet(path, index=True)
    else:
        frame.to_csv(path, index=True, compression='gzip')

def process_dataset(source, raw_dir, output_root, config):
    validate_settings(config)
    calendar = load_calendar(raw_dir, source['metadata']['start_date'], source['metadata']['end_date'])
    assert_source_unchanged(source)
    sample, train_rows, summary = collect_fit_sample(source, calendar, config)
    state = fit_preprocessors(sample, config)
    report = pd.DataFrame({
        'fit_missing_rate': sample.isna().mean(), 'tree_median': state['tree_median'],
        'linear_low': state['linear_low'], 'linear_high': state['linear_high'],
        'linear_mean': state['linear_mean'], 'linear_scale': state['linear_scale'],
    }, index=state['features'])
    report.index.name = 'feature'
    output_root = Path(output_root).resolve()
    output_root.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    run_id = f'run_{stamp}_{uuid.uuid4().hex[:8]}'
    staging, target = output_root / f'.{run_id}.tmp', output_root / run_id
    staging.mkdir()
    extension = '.parquet' if config['output_format'] == 'parquet' else '.csv.gz'
    saved_parts = {view: {split: 0 for split in ('train', 'valid', 'test')} for view in ('tree', 'linear')}
    try:
        assert_source_unchanged(source)
        for number, (path, X, info) in enumerate(iter_source(source, calendar, config), 1):
            for view in ('tree', 'linear'):
                transformed = transform_features(X, state, view)
                for split in ('train', 'valid', 'test'):
                    mask = info['split'].eq(split)
                    if not mask.any():
                        continue
                    folder = staging / view / split
                    folder.mkdir(parents=True, exist_ok=True)
                    table = transformed.loc[mask].join(info.loc[mask])
                    write_table(table, folder / f'part-{number:05d}{extension}', config['output_format'])
                    saved_parts[view][split] += 1
            print(f'处理上游分区 {number}/{len(source["paths"])}：{path.name}')
        assert_source_unchanged(source)
        metadata = {
            'schema_version': SCHEMA_VERSION, 'created_utc': stamp, 'format': config['output_format'],
            'target_name': label_name(config['target_horizon']), 'target_mode': config['target_mode'],
            'features': len(state['features']), 'views': ['tree', 'linear'],
            'fit_candidate_rows': int(train_rows), 'fit_rows': len(sample),
            'fit_method': 'all_train_rows' if config['fit_max_rows'] is None else 'smallest_index_hash_train_sample',
            'partitions': saved_parts, 'source_run': str(source['path']),
            'source_metadata_sha256': source['metadata_sha256'],
            'pandas': pd.__version__, 'numpy': np.__version__,
            'notebook_sha256': hashlib.sha256((PROJECT_DIR / 'labels&process.ipynb').read_bytes()).hexdigest()
                if (PROJECT_DIR / 'labels&process.ipynb').exists() else None,
            'label_formula': source['metadata'].get('label_formula'),
        }
        write_json(staging / 'config.json', config)
        write_json(staging / 'preprocessor.json', state)
        write_json(staging / 'feature_names.json', state['features'])
        write_json(staging / 'metadata.json', metadata)
        write_json(staging / 'source.json', {'path': str(source['path']),
                   'metadata_sha256': source['metadata_sha256'], 'partitions': source['manifest'],
                   'metadata': source['metadata'], 'calendar': calendar.strftime('%Y-%m-%d').tolist()})
        summary.to_csv(staging / 'split_summary.csv')
        report.to_csv(staging / 'training_feature_report.csv')
        staging.rename(target)
        pointer = output_root / f'.latest-{uuid.uuid4().hex}.tmp'
        write_json(pointer, {'run_dir': target.name})
        pointer.replace(output_root / 'latest_run.json')
    except Exception:
        if staging.is_dir() and staging.resolve().parent == output_root:
            shutil.rmtree(staging)
        raise
    display(summary)
    print(f'训练可用行 {train_rows:,}；统计量拟合样本 {len(sample):,}；特征 {len(state["features"])} 列')
    print(f'输出：{target}')
    return target

## 6. 模型读取接口

`load_model_data('train', view='tree') → (X, y)`，只返回标签、特征均可用的监督样本。
`load_prediction_data(split='test', latest_only=True) → (X, info)` 保留最新缺标签日期，
仅要求特征可用；info 保留原始收益和日期用于后续评价，不作为模型输入。

两个 view 的模型输入均为同序 158 列，索引为 `(datetime, instrument)`。
`iter_model_batches` 可逐批训练/预测；`load_*` 会拼接选中的数据，需注意内存。
`load_preprocessor` 和 `transform_features` 可在未来预测时复用保存状态。

In [ ]:
def resolve_dataset(run=None, root=OUTPUT_ROOT):
    if run is not None:
        return Path(run)
    root = Path(root)
    pointer = root / 'latest_run.json'
    if not pointer.exists():
        raise FileNotFoundError('尚无 model_dataset 成功输出，请先执行正式处理')
    path = (root / json.loads(pointer.read_text(encoding='utf-8'))['run_dir']).resolve()
    if path.parent != root.resolve():
        raise ValueError('数据集指针越出根目录')
    return path

def load_preprocessor(run=None):
    return json.loads((resolve_dataset(run) / 'preprocessor.json').read_text(encoding='utf-8'))

def iter_model_batches(split='train', view='tree', run=None, supervised=True, instruments=None):
    if split not in ('train', 'valid', 'test') or view not in ('tree', 'linear'):
        raise ValueError('split 或 view 无效')
    folder = resolve_dataset(run)
    metadata = json.loads((folder / 'metadata.json').read_text(encoding='utf-8'))
    names = json.loads((folder / 'feature_names.json').read_text(encoding='utf-8'))
    pattern = 'part-*.parquet' if metadata['format'] == 'parquet' else 'part-*.csv.gz'
    paths = sorted((folder / view / split).glob(pattern))
    if len(paths) != metadata['partitions'][view][split] or not paths:
        raise ValueError('模型数据分区缺失')
    for path in paths:
        if metadata['format'] == 'parquet':
            frame = pd.read_parquet(path)
        else:
            frame = pd.read_csv(path, dtype={'instrument': str}, parse_dates=['datetime', 'label_entry_date', 'label_end_date'])
            frame = frame.set_index(['datetime', 'instrument'])
        flag = 'supervised_eligible' if supervised else 'feature_eligible'
        # pandas 正常推断 bool；显式验证，避免字符串 'False' 被当作 True。
        if not frame[flag].isin([True, False]).all():
            raise ValueError(f'布尔标记格式无效：{path}')
        mask = frame[flag].eq(True)
        if instruments is not None:
            mask &= frame.index.get_level_values('instrument').isin(instruments)
        frame = frame.loc[mask].sort_index()
        X = frame[names].astype('float32')
        y = frame.target.copy()
        info = frame.drop(columns=names)
        if supervised and metadata['target_mode'] == 'direction':
            y = y.astype('int8')
        if supervised and y.isna().any():
            raise ValueError('监督数据存在缺标签行')
        yield X, y, info

def load_model_data(split='train', view='tree', run=None, instruments=None):
    batches = list(iter_model_batches(split, view, run, True, instruments))
    X = pd.concat([item[0] for item in batches]).sort_index()
    y = pd.concat([item[1] for item in batches]).sort_index()
    if X.empty or X.index.has_duplicates:
        raise ValueError('没有可用样本或索引重复')
    if not X.index.equals(y.index):
        raise ValueError('X / y 索引不一致')
    return X, y

def load_prediction_data(split='test', view='tree', run=None, latest_only=True, instruments=None):
    batches = list(iter_model_batches(split, view, run, False, instruments))
    X = pd.concat([item[0] for item in batches]).sort_index()
    info = pd.concat([item[2] for item in batches]).sort_index()
    if X.empty or X.index.has_duplicates:
        raise ValueError('没有可用预测样本或索引重复')
    if latest_only:
        latest = X.index.get_level_values('datetime').max()
        mask = X.index.get_level_values('datetime') == latest
        X, info = X.loc[mask], info.loc[mask]
    return X, info

## 7. 独立自检（无需正式因子输出）

检查交易日标签错位、跨分段清除、分类缺标签、训练抽样上限、全空和常数列、
参数 JSON 序列化，以及验证/测试极端值不影响训练拟合。
生成的示例数据只存在内存中，不冒充正式数据集。

In [ ]:
def self_checks():
    dates = pd.bdate_range('2022-01-03', periods=100)
    index = pd.MultiIndex.from_product([dates, ['AAA.SZ', 'BBB.SH']], names=['datetime', 'instrument'])
    rng = np.random.default_rng(158)
    frame = pd.DataFrame(rng.normal(size=(len(index), 158)), index=index, columns=EXPECTED_FEATURES)
    frame.iloc[:, 0] = np.nan
    frame.iloc[:, 1] = 1.
    date_series = pd.Series(dates, index=dates)
    frame['label_entry_date'] = date_series.shift(-1).reindex(index.get_level_values('datetime')).to_numpy()
    for horizon in (1, 5, 10):
        name = label_name(horizon)
        frame[f'{name}_end_date'] = date_series.shift(-(1+horizon)).reindex(index.get_level_values('datetime')).to_numpy()
        frame[name] = .01
        frame.loc[frame[f'{name}_end_date'].isna(), name] = np.nan
    cfg = make_config()
    cfg['splits'] = {'train': [str(dates[0].date()), str(dates[59].date())],
                     'valid': [str(dates[60].date()), str(dates[79].date())],
                     'test': [str(dates[80].date()), str(dates[-1].date())]}
    cfg.update(target_horizon=1, entry_lag=1, fit_max_rows=25)
    validate_settings(cfg)
    X, info = target_and_flags(frame, EXPECTED_FEATURES, dates, cfg)
    assert info.loc[(dates[58], 'AAA.SZ'), 'label_status'] == 'cross_boundary'
    assert not info.loc[(dates[59], 'AAA.SZ'), 'supervised_eligible']
    assert info.loc[(dates[57], 'AAA.SZ'), 'supervised_eligible']
    assert info.loc[(dates[-1], 'AAA.SZ'), 'feature_eligible']
    assert pd.isna(info.loc[(dates[-1], 'AAA.SZ'), 'target'])
    cfg_class = dict(cfg, target_mode='direction')
    _, classified = target_and_flags(frame, EXPECTED_FEATURES, dates, cfg_class)
    assert classified.loc[classified.supervised_eligible, 'target'].eq(1).all()
    assert pd.isna(classified.loc[(dates[-1], 'AAA.SZ'), 'target'])
    mask = info['split'].eq('train') & info.supervised_eligible
    subset = X.loc[mask]
    sampled = sample_training_rows(pd.DataFrame(), subset.iloc[:50], 25)
    sampled = sample_training_rows(sampled, subset.iloc[50:], 25)
    assert len(sampled) == 25
    assert sampled.index.isin(subset.index).all()
    expected = sample_training_rows(pd.DataFrame(), subset, 25)
    pd.testing.assert_frame_equal(sampled.sort_index(), expected.sort_index())
    state = fit_preprocessors(X.loc[mask], cfg)
    restored = json.loads(json.dumps(state, allow_nan=False))
    for view in ('tree', 'linear'):
        transformed = transform_features(X, restored, view)
        assert transformed.shape == X.shape and np.isfinite(transformed.to_numpy()).all()
        assert transformed.iloc[:, 0].eq(0).all()
    changed = frame.copy()
    changed.loc[changed.index.get_level_values('datetime') >= dates[60], EXPECTED_FEATURES] = 1e12
    changed_X, changed_info = target_and_flags(changed, EXPECTED_FEATURES, dates, cfg)
    changed_mask = changed_info['split'].eq('train') & changed_info.supervised_eligible
    assert state == fit_preprocessors(changed_X.loc[changed_mask], cfg)
    broken = frame.copy()
    broken.loc[(dates[0], 'AAA.SZ'), 'LABEL0_end_date'] = dates[3]
    try:
        target_and_flags(broken, EXPECTED_FEATURES, dates, cfg)
    except ValueError:
        pass
    else:
        raise AssertionError('未拒绝错位标签')
    print('通过：标签日期、跨界清除、分类缺标签、训练期预处理、全空/常数列、JSON 状态和未来扰动自检。')
    return {'checks': 'passed', 'features': 158, 'synthetic_rows': len(frame)}

self_check_result = self_checks()

## 8. 正式处理与结果预览

本单元处理第 2 节固定的上游版本。完成后，新模型统一读取此处输出。
如上游仍在下载或尚未生成因子，请先完成 `feature.ipynb`，再从第 2 节重新运行。
如果更改了第 1 节配置，应从头重新运行以刷新 config 和固定输入版本。

In [ ]:
dataset_run = None
if RUN_PROCESS and source is not None:
    dataset_run = process_dataset(source, RAW_DIR, OUTPUT_ROOT, config)
elif not RUN_PROCESS:
    print('RUN_PROCESS=False：已执行契约定义和独立自检，未构建正式数据。')
else:
    print('正式处理跳过：尚无成功 Alpha158 输出，请先运行 feature.ipynb。')

if dataset_run is not None:
    state = load_preprocessor(dataset_run)
    print(f'预处理已保存；全空训练列 {len(state["all_missing_features"])} 个。')
    for X_batch, y_batch, info_batch in iter_model_batches('train', 'tree', dataset_run):
        display(X_batch.head(3))
        display(y_batch.head(3))
        break

## 9. 后续模型调用示例

先在本 notebook 完成一次正式处理，随后在模型 notebook 中复用第 6 节接口。
`tree` 适用于 Random Forest / LightGBM 等模型；`linear` 适用于 Ridge 等需要缩放的模型。
比较模型时固定数据集运行目录、预测目标和评测区间。

```python
# 固定某次运行，避免后续 latest_run 改变而影响复现。
run = dataset_run
X_train, y_train = load_model_data('train', view='linear', run=run)
X_valid, y_valid = load_model_data('valid', view='linear', run=run)
X_test, y_test = load_model_data('test', view='linear', run=run)
X_latest, prediction_info = load_prediction_data('test', view='linear', run=run)
```

训练函数只接收 X 和 y；未来收益、可用标记和日期均不混入 X。
预测阶段加载保存的状态进行 transform，不能对验证、测试或未来数据再次 fit。
下一模块可在此基础上开展因子评价和模型训练；滚动训练时需为每个训练窗口单独拟合状态。